# Online Covariance and Correlation

Covariance and correlation matrices of several streams with `CovMatrix` from [OnlineStats.jl](https://github.com/joshday/OnlineStats.jl), read one row at a time, under incremental, rolling window and exponentially weighted memory.

In [ ]:
Pollis.packages("OnlineStats", "Plots")

In [ ]:
using OnlineStats, Random, Statistics, LinearAlgebra, Plots

Random.seed!(1)  # simulated: three variables whose true correlations are 0.6, 0.3 and 0.5
S = [1.0 0.6 0.3; 0.6 1.0 0.5; 0.3 0.5 1.0]  # the true covariance matrix, also the correlation matrix (unit variances)
vars = randn(10_000, 3) * cholesky(S).U  # one row per observation. Replace vars with your own matrix.

## Incremental Matrices

Each row updates running averages of the row and of its cross-products. `cov(o)` and `cor(o)` return the same stored matrix, so each result is copied before the next call.

In [ ]:
o = fit!(CovMatrix(), eachrow(vars))
C = copy(cov(o))
R = copy(cor(o))
println("Means:               ", round.(mean(o), digits=3))
println("Standard deviations: ", round.(sqrt.(var(o)), digits=3))
println("Correlation matrix:  ", round.(R, digits=3))

## Checking Against Batch Values

With equal weights the online matrices equal the batch ones up to rounding.

In [ ]:
println("Largest covariance difference:  ", maximum(abs.(C .- cov(vars))))
println("Largest correlation difference: ", maximum(abs.(R .- cor(vars))))

## Rolling Window and Exponential Weights

The rolling window keeps the last 500 rows as vectors; the exponentially weighted matrix keeps only averages, with span 500.

In [ ]:
window = fit!(MovingWindow(500, Vector{Float64}), collect.(eachrow(vars)))
rolling = fit!(CovMatrix(), value(window))
weighted = fit!(CovMatrix(weight=ExponentialWeight(500)), eachrow(vars))

println("Rolling correlation of columns 1 and 2:              ", round(cor(rolling)[1, 2], digits=3))
println("Exponentially weighted correlation of columns 1 and 2: ", round(cor(weighted)[1, 2], digits=3))

## A Correlation That Changes

For the first half the correlation of the two variables is 0.6; for the second half it is -0.4. The incremental correlation averages the two halves, while the exponentially weighted one, span 200, follows the change.

In [ ]:
z1 = randn(10_000); z2 = randn(10_000)
rho = vcat(fill(0.6, 5_000), fill(-0.4, 5_000))
pair = [z1 rho .* z1 .+ sqrt.(1 .- rho .^ 2) .* z2]  # two standard normal variables with correlation rho

equal = CovMatrix()
decaying = CovMatrix(weight=ExponentialWeight(200))
ce = Float64[]; cw = Float64[]
for x in eachrow(pair)
    fit!(equal, x); fit!(decaying, x)
    push!(ce, nobs(equal) > 1 ? cor(equal)[1, 2] : NaN)
    push!(cw, nobs(decaying) > 1 ? cor(decaying)[1, 2] : NaN)
end
plot([ce cw], label=["Incremental" "Exponentially Weighted"], xlabel="Observations", ylabel="Correlation")
plot!(rho, linestyle=:dash, label="True correlation")

## Merging Two Partitions

Matrices built on separate parts of the stream combine exactly with `merge!` under equal weights.

In [ ]:
a = fit!(CovMatrix(), eachrow(vars[1:5_000, :]))
b = fit!(CovMatrix(), eachrow(vars[5_001:end, :]))
merge!(a, b)
println("Largest difference from the batch correlation: ", maximum(abs.(cor(a) .- cor(vars))))